<a href="https://colab.research.google.com/github/SerchCM/LexiconJergasPeruanas/blob/main/Session02_Search_Agent_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- Warm-up: NumPy & Pandas ---
import numpy as np, pandas as pd

a = np.arange(12).reshape(3, 4)
print("Array:\n", a, "\nMean per column:", a.mean(axis=0))

df = pd.DataFrame({"score": [12, 15, 9, 18], "student": ["A","B","C","D"]})
print(df.describe())

Array:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]] 
Mean per column: [4. 5. 6. 7.]
           score
count   4.000000
mean   13.500000
std     3.872983
min     9.000000
25%    11.250000
50%    13.500000
75%    15.750000
max    18.000000


In [2]:
# --- 8-Puzzle: Node / Problem ---
import heapq, random, time
from itertools import count
from collections import deque

GOAL = (1,2,3,4,5,6,7,8,0)
MOVES = {"Up":(-1,0), "Down":(1,0), "Left":(0,-1), "Right":(0,1)}

class Problem:
    def __init__(self, initial, goal=GOAL):
        self.initial, self.goal = initial, goal

    def actions(self, state):
        r, c = divmod(state.index(0), 3)
        return [a for a,(dr,dc) in MOVES.items() if 0 <= r+dr < 3 and 0 <= c+dc < 3]

    def result(self, state, action):
        i = state.index(0); r, c = divmod(i, 3)
        dr, dc = MOVES[action]
        j = (r+dr)*3 + (c+dc)
        s = list(state); s[i], s[j] = s[j], s[i]
        return tuple(s)

    def goal_test(self, state):
        return state == self.goal

    def path_cost(self, cost, state1, action, state2):
        return cost + 1

    def h(self, node):
        dist = 0
        for idx, val in enumerate(node.state):
            if val == 0: continue
            gr, gc = divmod(val-1, 3); r, c = divmod(idx, 3)
            dist += abs(gr-r) + abs(gc-c)
        return dist

class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state, self.parent, self.action, self.path_cost = state, parent, action, path_cost
        self.depth = 0 if parent is None else parent.depth + 1

    def expand(self, problem):
        return [Node(problem.result(self.state, a), self, a,
                     problem.path_cost(self.path_cost, self.state, a, None))
                for a in problem.actions(self.state)]

    def solution(self):
        node, seq = self, []
        while node.parent:
            seq.append(node.action); node = node.parent
        return list(reversed(seq))

    def __lt__(self, other):
        return self.state < other.state

In [3]:
# --- BFS / DFS / A* ---
def bfs(problem):
    node = Node(problem.initial); expanded = 0
    if problem.goal_test(node.state): return node, expanded
    frontier, explored = deque([node]), {node.state}
    while frontier:
        node = frontier.popleft(); expanded += 1
        for child in node.expand(problem):
            if child.state not in explored:
                if problem.goal_test(child.state): return child, expanded
                explored.add(child.state); frontier.append(child)
    return None, expanded

def dfs(problem, limit=50):
    frontier, explored, expanded = [Node(problem.initial)], set(), 0
    while frontier:
        node = frontier.pop()
        if problem.goal_test(node.state): return node, expanded
        if node.state in explored or node.depth >= limit: continue
        explored.add(node.state); expanded += 1
        for child in node.expand(problem):
            if child.state not in explored: frontier.append(child)
    return None, expanded

def astar(problem):
    counter = count(); node = Node(problem.initial); expanded = 0
    frontier = [(problem.h(node), next(counter), node)]
    best_g = {node.state: 0}
    while frontier:
        _, _, node = heapq.heappop(frontier)
        if node.path_cost > best_g.get(node.state, float("inf")): continue
        expanded += 1
        if problem.goal_test(node.state): return node, expanded
        for child in node.expand(problem):
            if child.path_cost < best_g.get(child.state, float("inf")):
                best_g[child.state] = child.path_cost
                heapq.heappush(frontier, (child.path_cost + problem.h(child), next(counter), child))
    return None, expanded

In [4]:
# --- Shuffled start state ---
def is_solvable(state):
    t = [x for x in state if x != 0]
    return sum(1 for i in range(len(t)) for j in range(i+1, len(t)) if t[i] > t[j]) % 2 == 0

def scramble(n_moves=30, seed=1):
    rng = random.Random(seed); p = Problem(GOAL); s = GOAL
    for _ in range(n_moves):
        s = p.result(s, rng.choice(p.actions(s)))
    return s

start = scramble(30, seed=1)
print("Start:", start, "| solvable:", is_solvable(start))

Start: (0, 8, 3, 4, 7, 1, 2, 6, 5) | solvable: True


In [5]:
# --- Run the three algorithms on the same puzzle ---
rows = {}
for name, run in [("BFS", bfs), ("DFS", lambda p: dfs(p, 50)), ("A*", astar)]:
    t0 = time.perf_counter()
    node, expanded = run(Problem(start))
    rows[name] = {"Path length": len(node.solution()),
                  "Nodes expanded": expanded,
                  "Time (s)": round(time.perf_counter()-t0, 4)}

results = pd.DataFrame(rows).T
print(results, "\n")
print("BFS: optimal path, but expands every node level by level.")
print("DFS: reaches the goal deep in one branch, so the path is far from optimal.")
print("A*: same optimal path as BFS with far fewer nodes expanded.")

     Path length  Nodes expanded  Time (s)
BFS         22.0         59842.0    0.3334
DFS         50.0         22960.0    0.1145
A*          22.0           774.0    0.0080 

BFS: optimal path, but expands every node level by level.
DFS: reaches the goal deep in one branch, so the path is far from optimal.
A*: same optimal path as BFS with far fewer nodes expanded.
